# Retrieval-Augmented Generation (RAG)

In [2]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static nutrition table that we can use as a tool:

In [3]:
# We populated the RAG with the data from the data/questions_output.txt file in
# the rag_setup.ipynb notebook

chroma_client = chromadb.PersistentClient("../chroma")
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")

In [4]:
results = nutrition_qna.query(query_texts=["pregnancy"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

[('is_pregnancy', True)]
Question: What are some possible physical changes that a pregnant woman could experience in the middle months of gestation?
        Answer: During weeks 13-27, you may see an increase in weight and feel more hunger. Backaches might occur frequently, along with leg cramps and heartburn.

        This Q&A pair provides information about nutrition and health topics.


[('is_pregnancy', True)]
Question: What health issues can make a pregnancy more challenging?
        Answer: There are several medical conditions that can potentially increase the risk associated with Pregnancy. These include Anemia during this stage, Hypertensive Disorders related to gestation, Diabetes Mellitus coexisting with Pregnancy, Obesity during pregnancy, as well as Adolescent or Teenage pregnancies.

        This Q&A pair provides information about nutrition and health topics.




In [5]:
@function_tool
def nutrition_qna_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up questions and answers for nutrition data.

    Args:
        query: The question to lookup.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the question and answer result.
    """

    results = nutrition_qna.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No question data found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        question = metadata["Question"].title()
        answer = metadata["Answer"].title()

        formatted_results.append(
            f"{question}: {answer}"
        )

    return "Question and Answer:\n" + "\n".join(formatted_results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [6]:
nutrition_qna_tool('pregnancy')

TypeError: 'FunctionTool' object is not callable

In [8]:
nutrition_qna_agent = Agent(
    name="Nutrition Question and Answer",
    instructions="""
    You are a helpful nutrition assistant giving out answers to questions about nutrition.
    You give concise answers.
    If you need to look up answers to questions using the nutrition_qna_tool.
    """,
    tools=[nutrition_qna_tool]
)

In [10]:
with trace("Nutrition QnA Assistant with RAG"):
    result = await Runner.run(
        nutrition_qna_agent,
        "How many calories should I eat per day when pregnant",
    )
    print(result.final_output)

There isn’t a single number for everyone. Most pregnant people need about 300 extra calories per day in the second and third trimesters (little or no extra in the first trimester). Total daily needs often fall roughly in the 1,800–2,400 kcal range, depending on pre-pregnancy BMI and activity.

For a tailored estimate, share:
- pre-pregnancy weight and height (BMI)
- activity level
- any medical conditions

Always follow your healthcare provider’s guidance.
